## Load Libraries

In [1]:
import os
import json
import re
from getpass import getpass
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Set it in the environment or enter it when prompted."
    )

oa_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
OPENAI_MODEL = "gpt-4o-mini"
print(f"OpenAI client initialized with {OPENAI_MODEL}.")

OpenAI client initialized with gpt-4o-mini.


# Problem #1

## Initializing the prompt and the price of shoes

In [2]:
EMAIL_1 = '''Hey Jim,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 2 pair
2. Converse All-Star - 10 pair
3. New Balance 990 - 1 pair
4. Nike Zoom Fly 5 men's red - 2 pair

Please provide sub-totals and grand total cost.

Thanks.

Lance Gentry
123 Main St.
Chelsea, MI 48109
248-229-2229'''

EMAIL_2 = '''Hey Michelle,

I'd like to order some new shoes. Please ship the following:

1. Nike Air Jordan I - 1 pair
2. Converse All-Star - 20 pair
3. New Balance 990 - 2 pair
4. Nike Zoom Fly 5 men's red - 5 pair

Please provide sub-totals and grand total cost.

Thanks.

Artis Gilmore
723 Lexington Blvd.
New York, NY 10001
(503) 484-1029'''

# Fixed prices per pair, included in both prompts.
PRICE_LIST = {
    "Nike Air Jordan I": 180.00,
    "Converse All-Star": 60.00,
    "New Balance 990": 200.00,
    "Nike Zoom Fly 5 men's red": 160.00,
}
price_text = "\n".join(f"- {name}: ${price:.2f} per pair" for name, price in PRICE_LIST.items())
print(price_text)

- Nike Air Jordan I: $180.00 per pair
- Converse All-Star: $60.00 per pair
- New Balance 990: $200.00 per pair
- Nike Zoom Fly 5 men's red: $160.00 per pair


## Helper functions

call_llm sends a prompt to the selected chat model and returns its reply. extract_json removes optional Markdown code fences, finds the JSON object in the reply, and parses it into a Python object.

In [3]:
def call_llm(client, model, messages):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content


def extract_json(text):
    text = re.sub(r"```(?:json)?", "", text).strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("The model response did not contain a JSON object.")
    return json.loads(text[start:end + 1])

## Extract order from email and return json

This code asks the OpenAI model to extract Email 1’s order details and return them as JSON, including shoe prices, subtotals, and the grand total. It then parses the response into a Python object and prints it in a readable format.

In [4]:
zero_shot_prompt = f'''Extract the order from this email and return ONLY valid JSON.

Include shoe_agent_name, request_type, customer_name, customer_street,
customer_city, customer_state, customer_zip, customer_phone, and a shoes array.
Each shoe needs shoe_brand, shoe_model, shoe_quantity, shoe_price, and shoe_subtotal.
Also include grand_total. Use the listed prices and calculate subtotals and total.

Prices per pair:
{price_text}

Email:
{EMAIL_1}'''

zs_raw = call_llm(
    oa_client,
    OPENAI_MODEL,
    [{"role": "user", "content": zero_shot_prompt}],
)
zs_json = extract_json(zs_raw)
print(json.dumps(zs_json, indent=2))

{
  "shoe_agent_name": "Jim",
  "request_type": "order",
  "customer_name": "Lance Gentry",
  "customer_street": "123 Main St.",
  "customer_city": "Chelsea",
  "customer_state": "MI",
  "customer_zip": "48109",
  "customer_phone": "248-229-2229",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 2,
      "shoe_price": 180.0,
      "shoe_subtotal": 360.0
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 10,
      "shoe_price": 60.0,
      "shoe_subtotal": 600.0
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 1,
      "shoe_price": 200.0,
      "shoe_subtotal": 200.0
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 2,
      "shoe_price": 160.0,
      "shoe_subtotal": 320.0
    }
  ],
  "grand_total": 1480.0
}


## Use One shot prompt and json to extract from email2

This code builds a one-shot prompt: it shows the model Email 1 and its JSON result, then asks it to extract Email 2 in the same format using the listed prices. It sends the prompt to OpenAI, parses the reply as JSON, and prints it neatly.

In [12]:
one_shot_prompt = f'''Extract the order in the final email as JSON, using the same fields and format as the example.
Use the listed prices, calculate each shoe_subtotal, and calculate grand_total.
Return ONLY valid JSON.

Prices per pair:
{price_text}

Example email:
{EMAIL_1}

Correct example JSON:
{json.dumps(zs_json, indent=2)}

Email to extract:
{EMAIL_2}'''

os_raw = call_llm(
    oa_client,
    OPENAI_MODEL,
    [{"role": "user", "content": one_shot_prompt}],
)
os_json = extract_json(os_raw)
print(json.dumps(os_json, indent=2))

{
  "shoe_agent_name": "Michelle",
  "request_type": "order",
  "customer_name": "Artis Gilmore",
  "customer_street": "723 Lexington Blvd.",
  "customer_city": "New York",
  "customer_state": "NY",
  "customer_zip": "10001",
  "customer_phone": "(503) 484-1029",
  "shoes": [
    {
      "shoe_brand": "Nike",
      "shoe_model": "Air Jordan I",
      "shoe_quantity": 1,
      "shoe_price": 180.0,
      "shoe_subtotal": 180.0
    },
    {
      "shoe_brand": "Converse",
      "shoe_model": "All-Star",
      "shoe_quantity": 20,
      "shoe_price": 60.0,
      "shoe_subtotal": 1200.0
    },
    {
      "shoe_brand": "New Balance",
      "shoe_model": "990",
      "shoe_quantity": 2,
      "shoe_price": 200.0,
      "shoe_subtotal": 400.0
    },
    {
      "shoe_brand": "Nike",
      "shoe_model": "Zoom Fly 5 men's red",
      "shoe_quantity": 5,
      "shoe_price": 160.0,
      "shoe_subtotal": 800.0
    }
  ],
  "grand_total": 2580.0
}


# Problem #2: Chain-of-Thought Chat Completion

Recreate the textbook chat using OpenAI with the three worked examples as alternating user and assistant messages, followed by the age question. The examples show the model the expected problem-solving and answer style.

In [7]:
COT_EXAMPLES = [
    ("There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?",
     "There are 4 days from Monday to Thursday. 5 computers were added each day. That means in total 4 * 5 = 20 computers were added. There were 9 computers initially, so now there are 9 + 20 = 29 computers. The answer is 29."),
    ("Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?",
     "Michael initially had 58 balls. He lost 23 on Tuesday, so after that he has 58 - 23 = 35 balls. On Wednesday he lost 2 more so now he has 35 - 2 = 33 balls. The answer is 33."),
    ("Olivia has $23. She bought five bagels for $3 each. How much money does she have left?",
     "She bought 5 bagels for $3 each. This means she spent $15. She has $8 left."),
]
QUESTION = "When I was 6 my sister was half my age. Now I'm 70 how old is my sister?"

Build a chat history containing the three example questions and answers, then appends the sister-age question. It sends the full conversation to OpenAI and prints the model’s reply. The examples guide the model to solve the final problem in a similar step-by-step style.

In [9]:
v2_messages = [{"role": "system", "content": "You are a helpful assistant that answers math word problems."}]
for q, a in COT_EXAMPLES:
    v2_messages.append({"role": "user", "content": q})
    v2_messages.append({"role": "assistant", "content": a})
v2_messages.append({"role": "user", "content": QUESTION})

v2 = call_llm(oa_client, OPENAI_MODEL, v2_messages)
print(v2)

When you were 6, your sister was half your age, which means she was 3 years old (6 / 2 = 3). The age difference between you and your sister is 3 years. 

Now that you are 70, your sister is 70 - 3 = 67 years old. 

So, your sister is 67 years old now.


final_number finds all numbers in the model’s reply and returns the last one. The loop checks that answer from the multi-turn chat against the expected result, 67, and prints whether it matches.

In [11]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

for label, out in [ ("V2 multi-turn CoT", v2)]:
    ans = final_number(out)
    print(f"{label:24s} -> final number: {ans}  {'✅' if ans == '67' else '❌ (correct: 67)'}")

V2 multi-turn CoT        -> final number: 67  ✅


## Result and brief evaluation

The model answered **67**, which is correct. It identified that the sister was 3 when the speaker was 6, kept the 3-year age difference, and calculated $70 - 3 = 67$.

This follows the worked examples' use of arithmetic to reach an answer. The computer and golf-ball examples explicitly show each operation and end with “The answer is …”; the bagel example is shorter and leaves out the final-answer phrase. The model's reply used a concise explanation and stated the result clearly.

Compared with Problem 1, these responses required different skills. Problem 1 extracted customer and shoe details into JSON and calculated totals using a supplied price list; both orders' totals were correct. Problem 2 required interpreting the relationship over time rather than copying fields or summing line items. In both problems, the examples/instructions guided the response, but Problem 1's arithmetic was directly checkable from fixed prices, while Problem 2 depended on preserving the age difference.